# 00 — OECD data audit

**Purpose:** validate the supplied OECD file, identify time-series gaps, preserve pooled survey windows, and build same-year Australia comparisons.

The raw CSV is preserved unchanged. Reusable logic lives in `src/oecd_audit.py`; generated tables are written outside the notebook.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.oecd_audit import (
    build_australia_summary, build_gap_report, build_indicator_metadata,
    build_same_year_comparisons, load_clean, write_outputs,
)

DATA_FILE = PROJECT_ROOT / 'data' / 'raw' / 'OECD Data.csv'
DATA_FILE

In [ ]:
df = load_clean(DATA_FILE)
outputs = write_outputs(df)
print(f'{len(df):,} rows × {df.shape[1]:,} tidy columns')
print(f"{df['country_code'].nunique()} countries; {df['indicator_code'].nunique()} indicators")
display(df.head())

In [ ]:
metadata = build_indicator_metadata(df)
display(metadata)
print('Duplicate country-indicator-year keys:', df.duplicated(['country_code', 'indicator_code', 'year']).sum())
print('Flagged observations:', df['is_flagged'].sum())
display(df.groupby(['status_code', 'status'], dropna=False).size().rename('rows'))

## Australian gaps and non-annual designs

In [ ]:
gaps = build_gap_report(df)
aus_gaps = gaps.loc[gaps['country_code'].eq('AUS')]
display(aus_gaps[['indicator', 'frequency', 'first_year', 'last_year',
                  'observed_year_count', 'independent_period_count',
                  'calendar_missing_years', 'gap_interpretation']])

## Same-year comparisons and scouting summary

In [ ]:
comparisons = build_same_year_comparisons(df)
summary = build_australia_summary(df, comparisons, through_year=2024)
display(summary)
display(comparisons.loc[comparisons['comparison_is_thin']].tail(20))

## Interpretation rules

- Do not impute absent years merely to create a balanced panel.
- Treat PISA, elections and survey waves as periodic or event-based.
- Treat repeated Gallup pooled-window rows as one independent estimate.
- Compare countries only within the same indicator and year.
- Report the comparison-country count and OECD status flags.
- Read `docs/analysis/oecd_data_quality_notes.md` before interpreting ranks.